# Часть 2. Обучение классификатора тональности
**Датасет:** `MonoHime/ru_sentiment_dataset` — 3 класса: NEUTRAL (0), POSITIVE (1), NEGATIVE (2).

In [ ]:
!pip install -q datasets transformers torch scikit-learn seaborn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')

## 1. Загрузка данных

In [ ]:
df = load_dataset('MonoHime/ru_sentiment_dataset', split='train').to_pandas()
df = df[['text', 'sentiment']].dropna().rename(columns={'sentiment': 'label'})
df['label'] = df['label'].astype(int)

LABEL_NAMES = {0: 'NEUTRAL', 1: 'POSITIVE', 2: 'NEGATIVE'}

print(df.shape)
df.head(3)

In [ ]:
counts = df['label'].value_counts().sort_index()
print((counts / len(df)).rename('Доля').round(3).to_frame())

## 2. Подготовка данных

### 2.1 Разбивка на train / val / test

Используем стратифицированную разбивку, чтобы доля каждого класса сохранялась во всех частях. Для ускорения экспериментов берём 30 000 примеров: при таком размере модель достигает хорошего качества, а обучение занимает разумное время. Пропорция — 80 / 10 / 10.

In [ ]:
SAMPLE_SIZE = 30_000

df_sample = df.groupby('label', group_keys=False).apply(
    lambda g: g.sample(int(SAMPLE_SIZE * len(g) / len(df)), random_state=42)
).reset_index(drop=True)

train_df, tmp_df = train_test_split(df_sample, test_size=0.2, stratify=df_sample['label'], random_state=42)
val_df,   test_df = train_test_split(tmp_df,    test_size=0.5, stratify=tmp_df['label'],    random_state=42)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

### 2.2 Токенизация

In [ ]:
MODEL_NAME = 'cointegrated/rubert-tiny2'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt',
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }


train_ds = SentimentDataset(train_df['text'], train_df['label'])
val_ds   = SentimentDataset(val_df['text'],   val_df['label'])
test_ds  = SentimentDataset(test_df['text'],  test_df['label'])

print(f'Пример токенизации: {tokenizer.decode(train_ds[0]["input_ids"][:20])}')

## 3. Обучение классификатора

### 3.1 Выбор модели

Используется **`cointegrated/rubert-tiny2`** — компактная версия RuBERT (~29M параметров, ~115 МБ). Модель обучена на русских текстах и умеет строить контекстуальные представления слов — в отличие от TF-IDF она «понимает» порядок слов и многозначность.

Выбор именно tiny-версии обоснован практически: полный rubert-base (~180M параметров) потребовал бы в несколько раз больше времени на обучение и памяти, а качество на данной задаче при достаточном объёме данных отличается незначительно. Tiny-модель при этом хорошо сравнивается по скорости на CPU и GPU, что важно для оценки в пункте 4.

### 3.2 Метрика качества и функция потерь

**Метрика: F1-macro.**  
Датасет потенциально несбалансирован (нейтральный класс часто доминирует в агрегированных датасетах). Accuracy в таком случае даёт завышенный результат — высокий accuracy можно получить, просто предсказывая всегда самый частый класс. F1-macro усредняет F1 по каждому классу с одинаковым весом, что делает её чувствительной к ошибкам на всех классах.

**Функция потерь: Cross-entropy.**  
Стандартный выбор для многоклассовой классификации: она штрафует модель пропорционально уверенности в неправильном ответе и хорошо согласуется с softmax-выходом трансформера. При дисбалансе можно перейти на взвешенную cross-entropy (см. раздел 5).

### 3.3 Параметры обучения

| Параметр | Значение | Обоснование |
|---|---|---|
| learning_rate | 2e-5 | Стандартный диапазон для файнтюнинга BERT (2e-5 — 5e-5); меньшее значение снижает риск catastrophic forgetting |
| batch_size | 32 | Баланс между стабильностью градиентов и потреблением памяти |
| epochs | 3 | Большинство BERT-задач сходятся за 2–4 эпохи; больше — риск переобучения |
| warmup_ratio | 0.1 | Плавный разогрев lr на первых 10% шагов стабилизирует обучение |
| weight_decay | 0.01 | Лёгкая L2-регуляризация для предотвращения переобучения |

### 3.4 Обучение

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()},
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'f1_macro': f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds),
    }


training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 4. Оценка качества

### 4.1 Метрики на тестовой выборке

In [ ]:
predictions = trainer.predict(test_ds)
y_pred = predictions.predictions.argmax(axis=-1)
y_true = predictions.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=list(LABEL_NAMES.values()),
))

### 4.2 Матрица ошибок

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Абсолютные значения', 'Нормализованная (по строкам)'],
):
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=LABEL_NAMES.values(),
        yticklabels=LABEL_NAMES.values(),
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Предсказано')
    ax.set_ylabel('Истинный класс')

plt.tight_layout()
plt.show()

### 4.3 Скорость инференса: CPU vs GPU

Замеряем время обработки одного батча на каждом устройстве. Если GPU недоступен — сравниваем разные размеры батча на CPU.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE    = 32
N_WARMUP      = 3
N_MEASURE     = 20

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
sample_batch = next(iter(test_loader))


def measure_speed(model, batch, device, n_warmup=N_WARMUP, n_measure=N_MEASURE):
    model = model.to(device)
    model.eval()
    inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

    with torch.no_grad():
        for _ in range(n_warmup):
            model(**inputs)

    if device == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_measure):
            model(**inputs)
            if device == 'cuda':
                torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    avg_ms         = elapsed / n_measure * 1000
    samples_per_s  = BATCH_SIZE * n_measure / elapsed
    return avg_ms, samples_per_s


results = {}

cpu_ms, cpu_sps = measure_speed(model, sample_batch, 'cpu')
results['CPU'] = {'ms / batch': cpu_ms, 'samples / s': cpu_sps}
print(f'CPU  — {cpu_ms:.1f} мс/батч  |  {cpu_sps:.0f} примеров/с')

if torch.cuda.is_available():
    gpu_ms, gpu_sps = measure_speed(model, sample_batch, 'cuda')
    results['GPU'] = {'ms / batch': gpu_ms, 'samples / s': gpu_sps}
    print(f'GPU  — {gpu_ms:.1f} мс/батч  |  {gpu_sps:.0f} примеров/с')
else:
    print('GPU недоступен — сравнение только для CPU')

In [ ]:
if len(results) > 1:
    speed_df = pd.DataFrame(results).T

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, title in zip(
        axes,
        ['ms / batch', 'samples / s'],
        ['Время на батч (мс) — меньше лучше', 'Пропускная способность — больше лучше'],
    ):
        speed_df[col].plot(kind='bar', ax=ax, edgecolor='black', rot=0)
        ax.set_title(title)
        ax.set_ylabel(col)

    plt.tight_layout()
    plt.show()

    print(speed_df.round(1))
    if 'GPU' in results:
        speedup = results['CPU']['ms / batch'] / results['GPU']['ms / batch']
        print(f'\nGPU быстрее CPU в {speedup:.1f}x раз')
else:
    print(pd.DataFrame(results).T.round(1))

## 5. Варианты улучшения

**1. Использовать более крупную модель.**  
Текущая модель (`rubert-tiny2`) выбрана ради скорости обучения. Переход на `DeepPavlov/rubert-base-cased` или `ai-forever/ruBert-large` даёт заметный прирост F1, особенно на коротких и неоднозначных текстах. Это самый прямолинейный способ улучшить качество.

**2. Взвешенная функция потерь при дисбалансе классов.**  
Если нейтральный класс сильно преобладает, модель склонна предсказывать его слишком часто. Weighted cross-entropy назначает больший штраф за ошибки на редких классах. Веса можно задать обратно пропорционально частоте классов в train.

**3. Аугментация данных.**  
Для расширения обучающей выборки и снижения переобучения можно применить перефразирование через `back-translation` (русский → английский → русский) или замену слов синонимами с помощью тезауруса. Это особенно полезно для миноритарных классов.

**4. Квантизация модели для ускорения на CPU.**  
Квантизация переводит веса модели из float32 в int8, что уменьшает размер модели примерно в 4 раза и ускоряет инференс на CPU в 1.5–3 раза при минимальном снижении качества. Реализуется через `torch.quantization` или `optimum` от Hugging Face без переобучения.